# Apache Kafka Integration: Producers, Consumers, and Schema Handling (Optional)

**Module**: 09_streaming  
**Notebook**: 06_kafka_integration.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: How Spark Structured Streaming integrates with Apache Kafka: reading topics, consumer offsets, parsing key-value payloads, and writing to Kafka sinks.
- **Why does this topic exist?**: Kafka + Spark is the dominant real-time streaming architecture in enterprise tech.
- **What problem does it solve?**: Deserializing binary Kafka payloads and managing consumer offsets.
- **Where is it used in real-world Data Engineering?**: Enterprise event streams, IoT ingestion, and transaction event buses.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Advanced |
| **Estimated effort** | 3–4 h |
| **Prerequisites** | Streaming fundamentals, checkpoints, Kafka basics |
| **Required depth** | Working knowledge: can read from and write to Kafka with explicit offsets and failure handling |
| **Data Engineer** | Configure Kafka source/sink options and reason about delivery guarantees. |
| **Senior Data Engineer** | Diagnose offset, partition, and deserialization problems. |
| **Spark specialist** | Reason about consumer groups, rebalancing, and exactly-once with Kafka. |
| **Next topic** | `10_performance_and_optimization/01_spark_ui_analysis.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

Kafka is a natural source for Structured Streaming because it is replayable and partitioned. Spark commits
offsets to its checkpoint, not to Kafka's consumer group in the usual way, and the Kafka sink is at-least-once by
default. Exactly-once end-to-end therefore depends on idempotent downstream writes or a transactional sink.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **Bootstrap server** | Kafka endpoint | Initial connection for metadata |
| **Subscribe** | Topic subscription | Can use a topic list or pattern |
| **Starting offsets** | Where to begin | earliest, latest, or a JSON offset map |
| **maxOffsetsPerTrigger** | Rate limit | Bounds rows per micro-batch |
| **failOnDataLoss** | Missing-offset policy | Whether to fail when offsets are unavailable |
| **Kafka sink** | Write to a topic | At-least-once by default |

### How it works, step by step

1. The source connects to the bootstrap servers and discovers partitions.
2. It resolves starting offsets from the checkpoint or the configured option.
3. Each micro-batch fetches records up to `maxOffsetsPerTrigger`.
4. Records are deserialized from bytes into the declared schema.
5. The batch is processed and written to the sink.
6. Offsets are committed to the Spark checkpoint.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

## 2. Kafka + Spark Integration Architecture

```
Apache Kafka Cluster:
[Topic:'orders-stream', Partitions: 0, 1, 2]
                   |
                   v (1 Task per Kafka Partition!)
Spark Structured Streaming (Driver & Executors):
  - readStream.format("kafka").option("subscribe","orders-stream")
  - Each task connects directly to a Kafka Broker partition
                   |
                   v Deserialization & Business Logic (JSON / Avro)
  - df.select(from_json(col("value").cast("string"), schema))
                   |
                   v Output to Storage or Write Back to Kafka Sink
```

> **Note on Infrastructure**: This notebook provides complete production code examples. In environments without an active Kafka container broker, the code serves as an authoritative architectural reference.

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Kafka_Integration_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

## 3. Production Kafka Consumer Template

```python
# Reading from Kafka
kafka_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers","kafka:9092")
    .option("subscribe","orders_raw")
    .option("startingOffsets","latest") # or"earliest"
    .option("failOnDataLoss","false")
    .load()
)

# Kafka streams contain fixed schema: [key, value, topic, partition, offset, timestamp, timestampType]
parsed_stream = (
    kafka_stream
    .selectExpr("CAST(key AS STRING) as order_key","CAST(value AS STRING) as payload","timestamp")
    .select(F.col("order_key"), F.from_json(F.col("payload"), orders_schema).alias("order"),"timestamp")
    .select("order_key","order.*","timestamp")
)
```

---

## Experiment: Kafka Producer Specification

- **1. What are we testing?**: Explain how Spark writes to Kafka using key and value columns.
- **2. What do we expect?**: DataFrame with'key'and'value'columns written directly to Kafka topic.

In [ ]:
print("""
Writing to Kafka requires 'topic', 'key', and 'value' columns:
(
    df
    .selectExpr("CAST(order_id AS STRING) as key", "to_json(struct(*)) as value")
    .writeStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("topic", "orders_processed")
    .option("checkpointLocation", "/opt/spark-data/warehouse/chk_kafka_out")
    .start()
)
""")

**3. What happened?**: Displayed Kafka writer structure.  
**4. Why did it happen?**: Kafka expects byte arrays for key and value.  
**5. Practical Engineering Lesson**: Always serialize DataFrame records with to_json() or to_avro() before writing to Kafka.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: Real-time order events

- **Industry**: E-commerce
- **Scenario**: Order events flow through Kafka and must be aggregated for operational dashboards.
- **Problem**: The dashboard needs low latency and must recover from consumer restarts without losing events.
- **Why this concept is relevant**: Kafka's replayable log plus Spark checkpoints provides recoverable progress; an idempotent sink handles replay.
- **Implementation idea**: Read with `startingOffsets`, bound batch size, aggregate in event time, and write to an upsert-capable store.
- **Expected result**: Fresh aggregates with recoverable offsets.
- **Alternative**: Use Kafka Streams for the aggregation.
- **Trade-offs**: Kafka Streams keeps processing close to the data but introduces another runtime and skill set.

### Use Case 2: Dead-letter handling

- **Industry**: Payments
- **Scenario**: Some Kafka records cannot be deserialized or fail validation.
- **Problem**: Dropping them silently loses payment events; failing the whole stream blocks progress.
- **Why this concept is relevant**: A dead-letter topic preserves bad records while the main stream continues.
- **Implementation idea**: Parse with a permissive schema, split valid and invalid records, and write invalid ones to a DLQ with metadata.
- **Expected result**: No silent loss and an operable recovery path.
- **Alternative**: Fail fast and require manual intervention.
- **Trade-offs**: Failing fast protects correctness but can halt the pipeline; a DLQ balances progress and safety.

---

## Business Value

- Low-latency ingestion with replay.
- Recoverable offsets and bounded batches.
- Operable bad-record handling.
- Clear delivery semantics for downstream consumers.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Use a unique checkpoint per query.
- Set `maxOffsetsPerTrigger` to protect batch latency.
- Decide explicitly whether `failOnDataLoss` should fail or skip.
- Make the sink idempotent; the Kafka sink is at-least-once.
- Monitor consumer lag and batch duration.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Read Kafka with bounded batches

**Steps**

1. Produce a backlog of records to a topic.
2. Read with `maxOffsetsPerTrigger` set to a small value.
3. Observe batch input rows and duration, then increase the limit and compare.

**What to observe**: The limit bounds rows per micro-batch and protects latency at the cost of throughput.

**Healthy result**: Batch duration stays within the SLA.

**Problematic result**: A large backlog causes very long batches or memory pressure.

**Engineering takeaway**: Rate limiting is a latency-protection mechanism.

---

## Trade-offs and Alternatives

### Advantages

- Replayable source
- High throughput
- Partitioned parallelism

### Limitations

- Operational complexity
- At-least-once sink
- Offset/checkpoint management

### When to use it

- Event ingestion
- CDC
- Real-time dashboards

### When not to use it

- Assuming exactly-once without sink support
- Unbounded batches
- Ignoring consumer lag

### Alternatives inside Spark

- Kafka source/sink
- Checkpoints
- Watermarks

### Alternatives outside Spark

- Kafka Streams
- Flink
- Managed connectors

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Kafka Stream Schema
**Problem Statement**: What columns does readStream.format('kafka') return?  
**Input / Context**: Schema

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): failOnDataLoss Option
**Problem Statement**: Why set failOnDataLoss=false?  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Docker Compose Kafka Service
**Problem Statement**: Show how to add Kafka to Docker Compose.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Production Debugging Exercise

### Offsets out of range

**Scenario**: A stream fails after Kafka retention deletes the committed offsets.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Check the committed offset against the topic's earliest offset.
2. Decide whether to reset to earliest/latest or restore from a backup.
3. Review retention and checkpoint recovery procedures.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Kafka Stream Schema):
```python
print('key, value, topic, partition, offset, timestamp, timestampType')
```

#### Solution 2 (failOnDataLoss Option):
```python
print('Prevents stream failure if Kafka log retention purges older offsets before Spark finishes reading them.')
```

#### Solution 3 (Docker Compose Kafka Service):
```python
print('Add wurstmeister/kafka or confluentinc/cp-kafka service mapped to spark-net.')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Kafka Standard**: 1 Spark task per Kafka topic partition.
- **Serialization**: Values arrive as binary; cast to String and parse with from_json.

### Essential Technical Terms
- `bootstrap.servers`: Kafka broker connection string.
- `startingOffsets`: earliest or latest.

### Common Engineering Mistakes
- **Forgetting to cast binary value**: Attempting string operations directly on binary columns fails.

### Interview Questions on This Topic
- **Q: How is parallelism determined when reading from Kafka in Structured Streaming?**
  - *A*: Each partition of the Kafka topic is read by exactly one Spark task, so parallelism matches the number of Kafka topic partitions.


### More Interview Questions

- **Q: Where does Spark store Kafka offsets?**
  - *A*: In the query checkpoint, not in Kafka's consumer-group offsets by default; the checkpoint is used to resume.
- **Q: Is the Kafka sink exactly-once?**
  - *A*: No, it is at-least-once by default; exactly-once requires an idempotent or transactional downstream design.

### What to Study Next
Module 09 complete! Proceed to Module 10: Performance and Optimization.

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')